[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/17_dropout.ipynb)

# 🟢 Easy: Implement Dropout

Implement **Dropout** regularization from scratch.

### Signature
```python
class MyDropout(nn.Module):
    def __init__(self, p: float = 0.5): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Rules
- During **training**: zero each element with probability `p`, scale remaining by `1/(1-p)`
- During **eval**: return input unchanged (identity)
- Do NOT use `nn.Dropout` or `F.dropout`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

## Dropout
특정 뉴런에만 의존하는 현상을 없애기 위해 학습 중에 랜덤하게 뉴런을 끄는 것임
-> overfitting 방지 가능

1. 랜덤하게 끄기 (p=0.5 -> 50%의 뉴런을 끔)
2. 스케일 보정 (1/(1-확률) 을 살아남은 출력들에 곱해줌.) -> 왜 이 단계를 하냐? 학습 시 뉴런 절반이 꺼져있으니까 원래보다 출력이 작아짐. 반면 test때는 모든 뉴런이 커지니까 출력이 갑자기 커짐. 이 차이를 없애려고 학습할 때 미리 크기(scale)을 맞춰둠!!

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()

        # p는 __init__ 안에서만 살아있는 변수 -> forward에서도 쓰려면 이렇게 저장해줘야 함.
        self.p = p

    def forward(self, x):
        # test 모드라면 그냥 통과시킴
        # nn.Module에 내장된 변수.model.train()이면 True 됨
        if not self.training:
          return x

          # traning인 경우 - dropout 적용
          # p의 확률로 0, (1-p) 확률로 1인 마스크 생성
          # 0을 곱해준 애들은 dropout 됨 / 1 곱해주면 원래 값 그대로

        # torch.rand_like(x) - x랑 똑같은 shape로 0~1 사이 랜덤 숫자 채운 텐서 형성
        mask = (torch.rand_like(x) > self.p).float()

        # mask적용 + scale 보정
        return x * mask / (1 - self.p)

In [7]:
# 🧪 Debug
d = MyDropout(p=0.5)
d.train()
x = torch.ones(10)
print('Train:', d(x))
d.eval()
print('Eval: ', d(x))

Train: tensor([0., 0., 0., 0., 0., 2., 2., 2., 2., 0.])
Eval:  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('dropout')


🧪 Testing: Implement Dropout (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Eval mode is identity (0.5ms)
  ✅ [2/4] Training: zeros and scaling (22.7ms)
  ✅ [3/4] Drop rate is approximately p (1.9ms)
  ✅ [4/4] Gradient flow (17.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (42.7ms total)
  Progress saved. Run status() to see your dashboard.

